# API KULCS (sajátot adj meg!)

In [1]:
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

open_ai_api_key = config["open_ai_api_key"]

# Google Sheets tábla beolvasása

In [ ]:
import requests
import csv
from io import StringIO

sheet_id = "14HAoq-Ie3RSYfSl6h5bxqTL-fKhTp9r7vxTn6lcBChM"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

response = requests.get(url)
response.encoding = "utf-8"

reader = csv.reader(StringIO(response.text))
rows = list(reader)

for row in rows:
    print(row)

['lead_id', 'ceg_nev', 'iparag', 'bejovo_uzenet', 'tavalyi_arbevetel', 'tavalyi_profit']
['L001', 'A1 Kft.', 'ételrendelés / e-commerce', 'Szia! Napi 80-100 rendelésünk van, de a Google Ads költésünk mostanában nagyon elszaladt. Szeretnénk valakit, aki átnézi a kampányokat és javasol egy gyors optimalizálási tervet. Ha működik, havi együttműködésben gondolkodunk.', '185000000', '24000000']
['L002', 'A2 EV', 'lakberendezési webshop', 'Szia, most inditottuk a webshopot és van napi kb 5-10 látogató. érdekelne SEO meg hirdetés is, de még nincs fix büdzsé. Első körben csak szeretnénk látni nagyjából mibe kerülne egy ilyen együttműködés.', 'ez az első évem', '']
['L003', 'A3 Kft.', 'fitness terem', 'Két új edzőtermünk nyílik szeptemberben, és szeretnénk addigra leadeket gyűjteni próbaedzésekre. Meta és Google kampányban gondolkodunk, a nyitás miatt elég sürgős lenne. Kérlek hívjatok fel még ma vagy holnap.', '94000000', '8700000']
['L004', 'A4 Kft.', 'építőipari szolgáltatás', 'Évek óta aján

# Egyetlen sornyi adat kiértékelése OpenAI API hívással

In [3]:
import requests

row = rows[5]

prompt = f"""Értékeld a leadet 0-100 pont között.
A jó lead jellemzői:
* konkrét digitális marketing problémát ír le
* van üzleti célja: több lead, több webshop rendelés, jobb ROAS, jobb mérés, automatizált riport
* van sürgősség vagy döntési helyzet
* van pénzügyi kapacitásra utaló jel: árbevétel/profit/büdzsé/növekedés
* nem csak "érdeklődöm", hanem konkrét következő lépést kér
* digitális marketing ügynökség számára releváns szolgáltatást keres
* akkor is lehet jó lead, ha EV vagy kisebb cég, amennyiben konkrét és fizetőképes igénye van
A gyenge lead jellemzői:
* nincs konkrét probléma
* nincs büdzsé vagy nagyon kicsi a cég
* csak általános érdeklődés
* nem releváns szolgáltatást kér
* nem döntéshozó vagy nem derül ki a szándék
* a pénzügyi adatok hiányosak vagy bizonytalanok, és az üzenetből sem derül ki erős üzleti potenciál
* az üzenet nagyon igénytelen, zavaros vagy nehezen érthető, és emiatt nem látszik tisztán a valódi igény
Fontos:
* a cégnév anonimizált, abból ne következtess valódi cégméretre vagy minőségre
* az üzenet minősége fontos szempont legyen: a rossz helyesírás rossz leadre utal
* ha az árbevétel vagy profit mező üres, "nem tudom" vagy "ez az első évem", akkor a bejövő üzenet alapján próbáld megítélni az üzleti potenciált

Lead adatok:
- Iparág: {row[2]}
- Üzenet: {row[3]}
- Tavalyi árbevétel: {row[4]}
- Tavalyi profit: {row[5]}

Válaszolj csak így: PONTSZÁM: [szám] | INDOKLÁS: [1-2 mondat]"""

response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": "Bearer " + open_ai_api_key},
    json={
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": prompt}]
    }
)

print(response.json()["choices"][0]["message"]["content"])

PONTSZÁM: 85 | INDOKLÁS: A lead egy konkrét digitális marketing problémát fogalmaz meg a gyenge konverzióval kapcsolatban, világos üzleti céllal rendelkezik a vásárlók számának növelésére. A pénzügyi adatok is erősek, ami jelzi a cég kapacitását a szolgáltatás megfizetésére. A sürgősség és a konkrét igény mellett releváns szolgáltatást keresnek, ami még inkább megerősíti a lead értékét.


# Minden összefűzve, automatikus kiértékelése minden sornak a táblában

In [4]:
import requests
import csv
from io import StringIO

# --- STEP 1: Get leads from Google Sheet ---
sheet_id = "14HAoq-Ie3RSYfSl6h5bxqTL-fKhTp9r7vxTn6lcBChM"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

response = requests.get(url)
response.encoding = "utf-8"

reader = csv.reader(StringIO(response.text))
rows = list(reader)

# --- STEP 2: Evaluate each lead ---
for row in rows[1:]:   # skip header
    prompt = f"""Értékeld a leadet 0-100 pont között.
A jó lead jellemzői:
* konkrét digitális marketing problémát ír le
* van üzleti célja: több lead, több webshop rendelés, jobb ROAS, jobb mérés, automatizált riport
* van sürgősség vagy döntési helyzet
* van pénzügyi kapacitásra utaló jel: árbevétel/profit/büdzsé/növekedés
* nem csak "érdeklődöm", hanem konkrét következő lépést kér
* digitális marketing ügynökség számára releváns szolgáltatást keres
* akkor is lehet jó lead, ha EV vagy kisebb cég, amennyiben konkrét és fizetőképes igénye van
A gyenge lead jellemzői:
* nincs konkrét probléma
* nincs büdzsé vagy nagyon kicsi a cég
* csak általános érdeklődés
* nem releváns szolgáltatást kér
* nem döntéshozó vagy nem derül ki a szándék
* a pénzügyi adatok hiányosak vagy bizonytalanok, és az üzenetből sem derül ki erős üzleti potenciál
* az üzenet nagyon igénytelen, zavaros vagy nehezen érthető, és emiatt nem látszik tisztán a valódi igény
Fontos:
* a cégnév anonimizált, abból ne következtess valódi cégméretre vagy minőségre
* az üzenet minősége fontos szempont legyen: a rossz helyesírás rossz leadre utal
* ha az árbevétel vagy profit mező üres, "nem tudom" vagy "ez az első évem", akkor a bejövő üzenet alapján próbáld megítélni az üzleti potenciált

Lead adatok:
- Iparág: {row[2]}
- Üzenet: {row[3]}
- Tavalyi árbevétel: {row[4]}
- Tavalyi profit: {row[5]}

Válaszolj csak így: PONTSZÁM: [szám] | INDOKLÁS: [1-2 mondat]"""

    response = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": "Bearer " + open_ai_api_key},
        json={
            "model": "gpt-4o-mini",
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    evaluation = response.json()["choices"][0]["message"]["content"]
    print(f"{row[0]}: {evaluation}")

L001: PONTSZÁM: 85 | INDOKLÁS: A lead konkrét digitális marketing problémát fogalmaz meg, van világos üzleti célja (kampányoptimalizálás, havi együttműködés), és a pénzügyi adatok alapján jelentős üzleti potenciál is látható. A sürgősség is megjelenik, mivel a költések gyors átvizsgálását kérik.
L002: PONTSZÁM: 40 | INDOKLÁS: A lead érdeklődést mutat a digitális marketing iránt, de a pontos probléma leírása és a büdzsé hiánya miatt elégtelen. Az üzenet nem tartalmaz sürgősséget vagy konkrét következő lépést, ami csökkenti a potenciált.
L003: PONTSZÁM: 85 | INDOKLÁS: A lead konkrét digitális marketing problémát ír le, sürgősséget mutat a közelgő nyitás miatt, és világosan kér következő lépést. Az árbevétel és profit adatok is erős üzleti potenciálra utalnak, ami azt jelzi, hogy van pénzügyi kapacitásuk a szolgáltatások igénybevételére.


KeyboardInterrupt: 